# Análisis de pacientes Synthea con `clinlab`

Versión reescrita del notebook del laboratorio 2: la lógica (conversión de tipos, auditoría, uniones, clasificación, eGFR) vive en el paquete probado `clinlab` (`src/clinlab/`), y este notebook solo **lee los datos, llama a las funciones y explica los resultados**.

Requisitos: kernel del `.venv` con `pip install -e ".[dev,notebook]"` y los CSV de Synthea (`patients.csv`, `encounters.csv`, `observations.csv`) en la carpeta que indica `RUTA_DATOS` (sección 0).

## 0. Configuración

In [1]:
# --- Sección 0: configuración ---------------------------------------------
# Todo lo que depende de TU máquina o del entorno está en esta celda, y solo
# aquí. El resto del notebook no conoce rutas: solo usa RUTA_DATOS.

from pathlib import Path  # Path arma rutas que funcionan en Linux, Mac y Windows

import pandas as pd

# Las funciones vienen del paquete probado (src/clinlab), no se definen aquí.
# Si esta línea falla con ModuleNotFoundError, el kernel no es el del .venv
# (o falta `pip install -e ".[dev,notebook]"`).
from clinlab.auditoria import (
    contar_duplicados,
    detectar_fechas_imposibles,
    marcar_valores_implausibles,
)
from clinlab.clasificacion import clasificar_grupo_etario
from clinlab.clinico import calcular_egfr_ckd_epi
from clinlab.preprocesamiento import optimizar_dtypes
from clinlab.union import unir_con_validacion

# ÚNICA línea que hay que cambiar si tus CSV están en otro lado.
# Ruta relativa a esta carpeta (notebooks/): subir dos niveles
# (notebooks -> repo -> Data_Sciences) y entrar a data/. Relativa y no
# absoluta para no dejar /home/<usuario>/ escrito en un repo público.
RUTA_DATOS = Path("../../data")

# Los tres CSV de Synthea que usa este análisis (no están en el repo).
ARCHIVOS = ["patients.csv", "encounters.csv", "observations.csv"]

# Fallar TEMPRANO y con un mensaje claro, en vez de un FileNotFoundError a
# mitad del notebook, después de minutos de carga.
faltantes = [nombre for nombre in ARCHIVOS if not (RUTA_DATOS / nombre).exists()]
if faltantes:
    raise FileNotFoundError(
        f"No encuentro {faltantes} en {RUTA_DATOS.resolve()}. "
        "Cambia RUTA_DATOS en esta celda para que apunte a tus CSV de Synthea."
    )

# .resolve() convierte la ruta relativa en absoluta, para ver a dónde apunta.
print(f"Datos encontrados en: {RUTA_DATOS.resolve()}")

Datos encontrados en: /home/jorge/Documentos/Data_Sciences/data


## 1. Carga de datos

Cada tabla se lee **una sola vez** y se reutiliza en todo el notebook.

In [2]:
# --- Sección 1: carga de datos ---------------------------------------------
# Regla del notebook: cada CSV se lee UNA sola vez, aquí. Las secciones de
# abajo reutilizan estos DataFrames (el lab 2 releía observations varias veces
# y eso es lo que más tiempo y RAM costaba).
#
# Estrategia (decidida por el tamaño de cada archivo y el PICO de memoria):
#   - patients (7 MB) y encounters (444 MB): lectura "cruda" (todo como texto),
#     y en la sección 2 se optimizan con optimizar_dtypes. Así se puede medir
#     el antes/después Y se usa la función probada del paquete. El pico
#     (crudo + copia dentro de optimizar_dtypes) cabe de sobra en RAM.
#   - observations (3 GB): leerla cruda pesaría ~10 GB, y optimizar_dtypes
#     haría una copia encima. Por eso los tipos se piden YA en read_csv: pandas
#     construye cada columna directamente en su tipo final y nunca existe la
#     versión cruda completa en memoria.

# 1) Tablas chicas: lectura cruda, sin dtype ni parse_dates (a propósito).
pacientes_crudo = pd.read_csv(RUTA_DATOS / "patients.csv")
encuentros_crudo = pd.read_csv(RUTA_DATOS / "encounters.csv")

# 2) observations: optimizada desde la lectura.
#
# usecols: solo las columnas que el análisis realmente usa. Una columna que no
# se lee no ocupa memoria. Columnas disponibles:
#   DATE, PATIENT, ENCOUNTER, CATEGORY, CODE, DESCRIPTION, VALUE, UNITS, TYPE
# Pregunta para decidir: ¿qué secciones (3 a 6) usan cada columna? Si ninguna
# usa una, fuera.
COLUMNAS_OBSERVATIONS = [
    "UNITS",
    "VALUE",
    "ENCOUNTER",
    "DATE",
    "CODE",
    "DESCRIPTION",
]

# dtype: diccionario {columna: tipo}. Candidatas a "category": columnas con
# POCOS valores distintos que se repiten millones de veces (códigos, unidades,
# ids de paciente). Ojo con VALUE: mezcla números y texto; piensa si category
# es lo más útil para la auditoría de la sección 3 o si conviene dejarla texto.
TIPOS_OBSERVATIONS = {
    "UNITS": "category",
    "VALUE": "string",
    "ENCOUNTER": "category",
    "CODE": "category",
    "DESCRIPTION": "category",
}

observaciones = pd.read_csv(
    RUTA_DATOS / "observations.csv",
    usecols=COLUMNAS_OBSERVATIONS,
    dtype=TIPOS_OBSERVATIONS,
    parse_dates=["DATE"],
)

# Confirmación rápida de que las 3 tablas cargaron (filas, columnas).
for nombre, df in [
    ("patients", pacientes_crudo),
    ("encounters", encuentros_crudo),
    ("observations", observaciones),
]:
    print(f"{nombre:<13} {df.shape[0]:>12,} filas  {df.shape[1]:>3} columnas")

patients            22,888 filas   28 columnas
encounters       1,330,839 filas   15 columnas
observations    16,964,801 filas    6 columnas


## 2. Optimización de memoria

### Actividad 1: Análisis inicial de patients.csv

Antes de realizar la carga con tipos explícitos, se clasifican las columnas de `patients.csv` para determinar los `dtypes`:
* **Identificadores (Id):** La columna `Id` es un UUID (cadena alfanumérica única). Debe quedarse como `object` o cadena de texto, ya que cada valor es único y no tiene sentido agruparlo.
* **Fechas (Dates):** Las columnas `BIRTHDATE` y `DEATHDATE` deben parsearse directamente como `datetime64` (en pandas 3, `datetime64[us]`) para poder calcular edades o secuencias temporales.
* **Categóricas:** Columnas como `MARITAL` (estado civil), `RACE`, `ETHNICITY`, `GENDER`, `CITY` y `STATE` toman valores repetitivos de un conjunto cerrado. Se deben cargar como `category` para ahorrar memoria.

In [3]:
# --- Sección 2: optimización de memoria --------------------------------------
# Aquí se convierten las dos tablas que se leyeron "crudas" en la sección 1
# (patients y encounters) con optimizar_dtypes, la función probada del paquete,
# y se mide cuánta memoria se ahorró columna por columna.
# observations NO pasa por aquí: ya se leyó con sus tipos finales (sección 1).


def memoria_mb(df: pd.DataFrame) -> pd.Series:
    """MB que ocupa cada columna de df."""
    # deep=True es CLAVE: sin él, pandas solo cuenta los punteros de las
    # columnas de texto (8 bytes por fila) y no el texto real, así que el
    # "antes" saldría muchísimo más chico de lo que de verdad pesa.
    # index=False: solo interesan las columnas, no el índice.
    return df.memory_usage(deep=True, index=False) / 1024**2  # bytes -> MB


# 1) Qué columnas convertir en cada tabla.
#    Criterio (el mismo de la Actividad 1, en la celda de arriba):
#      - category: pocos valores distintos que se repiten mucho.
#      - fecha: columnas de fecha que llegaron como texto.
#      - ids únicos (Id, SSN...) y texto libre (ADDRESS...) se quedan como están:
#        como category no ahorran nada (una categoría por fila).
#    Tip: pacientes_crudo.nunique() te dice cuántos valores distintos tiene cada
#    columna; úsalo para decidir, no solo la intuición.
CATEGORICAS_PACIENTES = [
    "GENDER",
    "RACE",
    "ETHNICITY",
    "MARITAL",
]
FECHAS_PACIENTES = [
    "BIRTHDATE",
    "DEATHDATE",
]
CATEGORICAS_ENCUENTROS = [
    "PATIENT",
    "ENCOUNTERCLASS",
    "PAYER",
    "CODE",
    "DESCRIPTION",
    "REASONCODE",
    "REASONDESCRIPTION",
    "ORGANIZATION",
    "PROVIDER",
]
FECHAS_ENCUENTROS = [
    "START",
    "STOP",
]

# 2) Convertir. optimizar_dtypes devuelve una COPIA (no modifica el crudo),
#    así que los dos existen a la vez y podemos compararlos.
pacientes = optimizar_dtypes(pacientes_crudo, CATEGORICAS_PACIENTES, FECHAS_PACIENTES)
encuentros = optimizar_dtypes(
    encuentros_crudo, CATEGORICAS_ENCUENTROS, FECHAS_ENCUENTROS
)


# 3) Tabla comparativa por columna (para encounters, la tabla más grande que
#    tenemos en versión cruda; observations no tiene "antes", ver markdown).
#    pd.DataFrame con un diccionario: cada clave es una columna de la tabla y
#    cada valor una Series indexada por nombre de columna, así que pandas
#    alinea todo solo por el nombre de la columna.
comparacion = pd.DataFrame(
    {
        "dtype antes": encuentros_crudo.dtypes,
        "dtype después": encuentros.dtypes,
        "MB antes": memoria_mb(encuentros_crudo),
        "MB después": memoria_mb(encuentros),
    }
)
comparacion["ahorro %"] = (
    (comparacion["MB antes"] - comparacion["MB después"]) / comparacion["MB antes"]
) * 100

# .round(2) solo para que se lea mejor; no cambia los datos guardados.
display(comparacion.round(2))

# 4) Resumen por tabla (total en MB).
mb_pacientes_antes = memoria_mb(pacientes_crudo).sum()
mb_pacientes_despues = memoria_mb(pacientes).sum()

mb_encuentros_antes = memoria_mb(encuentros_crudo).sum()
mb_encuentros_despues = memoria_mb(encuentros).sum()

mb_observaciones = memoria_mb(observaciones).sum()

print(f"patients:     {mb_pacientes_antes:.2f} MB -> {mb_pacientes_despues:.2f} MB")
print(f"encounters:   {mb_encuentros_antes:.2f} MB -> {mb_encuentros_despues:.2f} MB")
print(f"observations: {mb_observaciones:.2f} MB (final optimizada)")

# 5) Liberar memoria: las versiones crudas ya no se usan en el resto del
#    notebook. `del` borra el NOMBRE; si nada más apunta a ese DataFrame,
#    Python libera su memoria.
del pacientes_crudo, encuentros_crudo

,dtype antes,dtype después,MB antes,MB después,ahorro %
Id,str,str,107.88,107.88,0.00
START,str,"datetime64[us, UTC]",87.57,10.15,88.41
STOP,str,"datetime64[us, UTC]",87.57,10.15,88.41
PATIENT,str,category,107.88,4.39,95.93
ORGANIZATION,str,category,107.88,2.63,97.56
PROVIDER,str,category,107.88,2.63,97.56
PAYER,str,category,107.88,1.27,98.82
ENCOUNTERCLASS,str,category,74.20,1.27,98.29
CODE,int64,category,10.15,1.27,87.50
DESCRIPTION,str,category,107.51,1.27,98.81


patients:     26.83 MB -> 20.67 MB
encounters:   1028.78 MB -> 178.48 MB
observations: 1285.24 MB (final optimizada)


### Resultados de la optimización (encounters)

La comparación columna por columna se hace sobre `encounters.csv`, la tabla más grande que se leyó sin tipos. `observations.csv` no tiene "antes": se leyó directo con sus tipos finales (sección 1) para no pasar por los ~10 GB de la lectura ingenua.

| Columna | dtype antes | dtype después | MB antes | MB después | Ahorro | Qué se pierde |
| :--- | :--- | :--- | ---: | ---: | ---: | :--- |
| **Id** | `str` | `str` | 107.88 | 107.88 | 0 % | **Nada:** no se convirtió. Es único por fila, así que como `category` no ahorraría nada (una categoría por fila). |
| **START / STOP** | `str` | `datetime64[us, UTC]` | 87.57 c/u | 10.15 c/u | 88 % | **El texto original** (formato exacto del CSV). Además quedan *con zona horaria* (UTC): compararlas con fechas sin zona, como `BIRTHDATE`, lanza `TypeError` (ver sección 3). |
| **PATIENT** | `str` | `category` | 107.88 | 4.39 | 96 % | **El ahorro, al unir tablas:** al hacer merge contra `patients.Id` (texto), pandas devuelve la llave como texto y el ahorro desaparece en el resultado. |
| **ORGANIZATION / PROVIDER** | `str` | `category` | 107.88 c/u | 2.63 c/u | 98 % | **Asignar valores nuevos:** escribir un valor que no está entre las categorías lanza `TypeError`; primero hay que agregarlo con `.cat.add_categories()`. |
| **PAYER / ENCOUNTERCLASS** | `str` | `category` | 107.88 / 74.20 | 1.27 c/u | 98-99 % | **Lo mismo que ORGANIZATION.** Con solo 10 valores distintos cada una, es el caso ideal de `category`. |
| **CODE** | `int64` | `category` | 10.15 | 1.27 | 88 % | **Operaciones numéricas y de orden:** `sum()` y comparaciones `<`/`>` lanzan `TypeError` (categoría sin orden). No importa: es una *etiqueta*, solo se filtra con `==`. |
| **DESCRIPTION / REASONDESCRIPTION** | `str` | `category` | 107.51 / 81.76 | 1.27 / 2.55 | 97-99 % | **Poco:** los métodos `.str` siguen funcionando, pero su resultado vuelve a ser texto (`object`), sin el ahorro. |
| **REASONCODE** | `float64` | `category` | 10.15 | 2.54 | 75 % | **Nada relevante:** era `float64` solo porque tiene faltantes (`NaN`); como `category` el faltante se conserva como faltante. |
| **BASE_ENCOUNTER_COST, TOTAL_CLAIM_COST, PAYER_COVERAGE** | `float64` | `float64` | 10.15 c/u | 10.15 c/u | 0 % | **Nada:** no se convirtieron. Son *cantidades* con las que se calcula; como `category` se perderían `sum()`, `mean()`, etc. |

**Totales:** encounters pasa de 1028.78 MB a 178.48 MB (−83 %); patients de 26.83 MB a 20.67 MB (−23 %); observations ocupa 1285.24 MB ya optimizada.

## 3. Auditoría de calidad de datos

In [4]:
# --- Sección 3: auditoría de calidad de datos --------------------------------
# Cuatro preguntas, en el mismo orden que los hallazgos del markdown de abajo:
#   1) ¿Cuántos faltantes hay?  2) ¿Hay ids duplicados?
#   3) ¿Hay fechas imposibles?  4) ¿Qué tan "numérica" es VALUE?


# 1) Faltantes: % de valores nulos por columna.
def porcentaje_faltantes(df: pd.DataFrame) -> pd.Series:
    """% de NaN por columna, solo las que tienen alguno, de mayor a menor."""
    # .isna() da True/False por celda; .mean() de booleanos = proporción de
    # True (True cuenta como 1), así que * 100 ya es el porcentaje.
    porcentaje = df.isna().mean() * 100
    # Mostrar solo columnas con faltantes: las de 0 % no aportan al análisis.
    return porcentaje[porcentaje > 0].sort_values(ascending=False).round(2)


print("Faltantes en patients (%):")
display(porcentaje_faltantes(pacientes))

print("\nFaltantes en encounters (%):")
display(porcentaje_faltantes(encuentros))

print("\nFaltantes en observations (%):")
display(porcentaje_faltantes(observaciones))


# 2) Duplicados en la llave primaria de patients.
# 2) Duplicados en la llave primaria de patients.
duplicados = contar_duplicados(pacientes["Id"])

print(f"Duplicados: {duplicados}")


# 3) Fechas imposibles: visitas antes del nacimiento.
# detectar_fechas_imposibles compara dos Series ALINEADAS fila a fila, pero la
# fecha de visita (START) está en encuentros y la de nacimiento en pacientes.
# Hay que "traer" el BIRTHDATE de cada paciente a cada fila de encuentros:
#   - pacientes.set_index("Id")["BIRTHDATE"] es una Series "Id -> nacimiento",
#     como un diccionario.
#   - .map(...) reemplaza cada PATIENT de encuentros por su nacimiento.
# Resultado: una Series con el mismo largo e índice que encuentros.
nacimiento_por_visita = encuentros["PATIENT"].map(
    pacientes.set_index("Id")["BIRTHDATE"]
)

# Zona horaria: START es datetime64[us, UTC] y BIRTHDATE no tiene zona, así que
# compararlas lanza TypeError. Se resuelve aquí y no dentro de la función: en qué
# zona están las fechas depende de cada dataset, y una función genérica no debe
# adivinarlo.
# .tz_localize(None) permite borrar una etiqueta y deja la hora tal cual
inicio_visita = encuentros["START"].dt.tz_localize(None)

visitas_antes_de_nacer = detectar_fechas_imposibles(
    inicio_visita, nacimiento_por_visita
)
# .sum() de una Series booleana = cuántos True (cuántas visitas imposibles).
print(f"Visitas antes del nacimiento: {visitas_antes_de_nacer.sum():,}")


# --- Visitas DESPUÉS de la muerte (DEATHDATE) ---
muerte_por_visita = encuentros["PATIENT"].map(pacientes.set_index("Id")["DEATHDATE"])

visitas_despues_de_morir = inicio_visita > muerte_por_visita

print(f"Visitas después de la muerte: {visitas_despues_de_morir.sum():,}")


# 4) VALUE: ¿cuántas observaciones NO son numéricas?
# pd.to_numeric(..., errors="coerce") convierte lo que puede y pone NaN en lo
# que no es número (ej. "Never smoker"), en vez de lanzar un error.

valores_numericos = pd.to_numeric(observaciones["VALUE"], errors="coerce")

es_texto = valores_numericos.isna() & observaciones["VALUE"].notna()

conteo_texto = es_texto.sum()
total_observaciones = len(observaciones)
porcentaje_texto = (conteo_texto / total_observaciones) * 100

print(f"Observaciones no numéricas (texto): {conteo_texto:,}")
print(f"Porcentaje sobre el total: {porcentaje_texto:.2f}%")

Faltantes en patients (%):


SUFFIX       98.90
DEATHDATE    87.38
MAIDEN       72.78
MARITAL      32.72
FIPS         25.76
PASSPORT     21.84
MIDDLE       19.76
PREFIX       19.41
DRIVERS      16.93
dtype: float64


Faltantes en encounters (%):


REASONCODE           36.97
REASONDESCRIPTION    36.97
dtype: float64


Faltantes en observations (%):


UNITS        27.45
ENCOUNTER     3.69
dtype: float64

Duplicados: 0
Visitas antes del nacimiento: 0
Visitas después de la muerte: 3,171
Observaciones no numéricas (texto): 6,253,652
Porcentaje sobre el total: 36.86%


### 3. Auditoría de Calidad de Datos (Hallazgos)

**1. Análisis de Valores Faltantes**
* **patients.csv:** La alta tasa de valores nulos en `DEATHDATE` (87.38%) es un comportamiento lógico y esperado, ya que la gran mayoría de los pacientes en la simulación siguen vivos. Las ausencias en columnas como `SUFFIX` (98.90%) o `MAIDEN` (apellido de soltera, 72.78%) son normales debido a la naturaleza opcional de esos datos demográficos.
* **encounters.csv:** Existe un 36.97% de valores faltantes en `REASONCODE` y `REASONDESCRIPTION`. Esto sugiere que no todas las visitas médicas tienen un código de diagnóstico principal explícito (pueden ser visitas administrativas, check-ups de rutina o seguimientos donde no se emite un nuevo diagnóstico).
* **observations.csv:** Destaca un 27.45% de falta de datos en la columna `UNITS`. Este faltante tiene sentido y está correlacionado con los resultados del punto 4, ya que las mediciones de tipo categórico o cualitativo no llevan unidades físicas.
* **observations.csv:** Además, el 3.69% de las observaciones no tiene `ENCOUNTER`, es decir, no está asociado a ninguna visita. Esas filas no tendrán con qué unirse en el Merge 1 (sección 4).

**2. Duplicidad de Identificadores**
* El dataset tiene una integridad referencial limpia en su tabla principal: se encontraron **0 duplicados** en la llave primaria de `patients.csv`.

**3. Coherencia Temporal (Anomalías Lógicas)**
* **Positivo:** No existen registros de encuentros médicos anteriores a la fecha de nacimiento.
* **Anomalía explicada:** Se detectaron **3,171 encuentros clínicos posteriores a la fecha de defunción** del paciente. Aunque parece un error, en las bases de datos de salud (y modelado por Synthea) esto representa actividad administrativa post-mortem legítima, como la emisión de certificados de defunción, trámites de facturación, o reportes de autopsias.

**4. Naturaleza de la columna VALUE en observations.csv**
* De un total de 16,964,801 observaciones, el **36.86% (6,253,652 registros) no son convertibles a formato numérico**.
* **Diagnóstico:** Al inspeccionar estas filas, confirmamos que la columna `VALUE` está "sobrecargada". Almacena simultáneamente mediciones continuas (que sí son números) y respuestas de encuestas sociales o categóricas (ej. estado de tabaquismo, orientación sexual, estatus de vivienda).
* **Conclusión:** Forzar la conversión de esta columna a tipos de datos numéricos flotantes (`float32` o `float64`) para ahorrar memoria provocaría una pérdida de información destructiva, eliminando más de un tercio del historial clínico cualitativo de los pacientes.

## 4. Uniones

### Actividad 4: Uniones y Validación de Cardinalidades

**Predicción del Merge 1: observations → encounters**
* **Cardinalidad esperada:** Muchos a uno (`m:1`). En una sola visita médica (encuentro) se le pueden hacer múltiples observaciones a un paciente (ej. medir peso, altura y presión arterial). Por lo tanto, muchas observaciones apuntan a un solo registro de encuentro.
* **Filas esperadas:** El resultado debe tener exactamente el mismo número de filas que la tabla `observations` original. Ninguna observación debe duplicarse.

**Predicción del Merge 2: tabla combinada → patients**
* **Cardinalidad esperada:** Muchos a uno (`m:1`). Un paciente a lo largo de su vida genera miles de observaciones clínicas en múltiples encuentros. Muchas filas del lado izquierdo apuntarán a un único registro maestro de paciente en el lado derecho.
* **Filas esperadas:** Exactamente el mismo número que resultó del Merge 1. No deben generarse filas extra.

In [5]:
# --- Sección 4: uniones -------------------------------------------------------
# Dos merges m:1 con unir_con_validacion (la función del paquete): si la
# cardinalidad real no es m:1, pandas lanza MergeError y el notebook se detiene
# en vez de seguir con filas multiplicadas en silencio.
#   Merge 1: observaciones (muchas)  -> encuentros (una visita)
#   Merge 2: resultado del merge 1   -> pacientes (un paciente)

# 1) Elegir columnas de la tabla derecha ANTES de unir.
#    - Memoria: el resultado tiene ~17 millones de filas; cada columna extra
#      que arrastres ocupa espacio en TODAS esas filas.
#    - Nombres repetidos: observaciones y encuentros tienen ambas CODE y
#      DESCRIPTION. Si traes las dos, pandas las renombra CODE_x / CODE_y y
#      ya no se sabe cuál es cuál.
#    Pregunta para decidir: ¿qué columnas de encuentros necesitan las
#    secciones 5 y 6? Como mínimo, la llave y lo que sirva para llegar al
#    paciente.
COLUMNAS_ENCUENTROS = [
    "Id",
    "PATIENT",
]

filas_antes = len(observaciones)

# how="left" y validate="m:1" son los valores por defecto de la función, por
# eso no se escriben: left conserva TODAS las observaciones aunque no crucen.
observaciones_encuentros = unir_con_validacion(
    observaciones,
    encuentros[COLUMNAS_ENCUENTROS],
    llave_izquierda="ENCOUNTER",
    llave_derecha="Id",
)

# Con left + m:1 el número de filas NO debe cambiar. Un assert en el notebook
# es un "mini test": si algún día falla, el notebook se detiene aquí mismo.
assert len(observaciones_encuentros) == filas_antes, "¡El merge 1 cambió filas!"
print(f"Merge 1: {filas_antes:,} -> {len(observaciones_encuentros):,} filas")

sin_visita = observaciones_encuentros["PATIENT"].isna()
print(
    f"Observaciones sin visita: {sin_visita.sum():,} ({sin_visita.mean() * 100:.2f}%)"
)

# Tras unir, "Id" (de encuentros) repite lo mismo que ENCOUNTER, y además
# chocaría con "Id" de pacientes en el merge 2 (-> Id_x / Id_y). Se descarta.
observaciones_encuentros = observaciones_encuentros.drop(columns="Id")


# 2) Merge 2: contra pacientes.
COLUMNAS_PACIENTES = [
    "Id",
    "BIRTHDATE",
    "GENDER",
]
filas_antes_merge2 = len(observaciones_encuentros)

tabla_completa = unir_con_validacion(
    observaciones_encuentros,
    pacientes[COLUMNAS_PACIENTES],
    llave_izquierda="PATIENT",
    llave_derecha="Id",
)

# Assert de consistencia: un left join validado m:1 no debe alterar el total de filas
assert len(tabla_completa) == filas_antes_merge2, "¡El merge 2 cambió filas!"
print(f"Merge 2: {filas_antes_merge2:,} -> {len(tabla_completa):,} filas")

# Auditoría del merge 2: observaciones que no tienen datos de pacientes
sin_paciente = tabla_completa["BIRTHDATE"].isna()
print(
    f"Observaciones sin paciente: {sin_paciente.sum():,} "
    f"({sin_paciente.mean() * 100:.2f}%)"
)

Merge 1: 16,964,801 -> 16,964,801 filas
Observaciones sin visita: 625,620 (3.69%)
Merge 2: 16,964,801 -> 16,964,801 filas
Observaciones sin paciente: 625,620 (3.69%)


In [6]:
observaciones.loc[sin_visita, "DESCRIPTION"].value_counts().head(10)

DESCRIPTION
DALY                                                                                              208540
QALY                                                                                              208540
QOLS                                                                                              208540
Abuse Status [OMAHA]                                                                                   0
Activated clotting time (ACT) of Blood by Coagulation assay                                            0
Address                                                                                                0
Adenovirus A+B+C+D+E DNA [Presence] in Respiratory system specimen by NAA with probe detection         0
Alanine aminotransferase [Enzymatic activity/volume] in Serum or Plasma                                0
Albumin [Mass/volume] in Serum or Plasma                                                               0
Alkaline phosphatase [Enzymatic activity/vo

### 4. Auditoría de Unión (Resultados)

**Merge 1: observations → encounters**
* **Validación de cardinalidad:** Pasó exitosamente (`m:1`). Las filas se mantuvieron constantes en 16,964,801, confirmando que la relación era correcta y no hubo un producto cartesiano (multiplicación de filas).
* **Auditoría de cruce:** `unir_con_validacion` no expone `indicator=True`, así que las filas sin cruce se contaron como las que quedaron con `PATIENT` vacío (esa columna viene de `encounters`, donde nunca es nula). Resultado: **625,620 observaciones (3.69 %) sin visita**, exactamente las que tienen `ENCOUNTER` faltante en la sección 3. No es un error de unión: ninguna observación *con* visita quedó sin cruzar.
* **Qué son esas filas:** solo tres tipos, con 208,540 registros cada uno: `DALY`, `QALY` y `QOLS`, indicadores de calidad de vida que Synthea calcula por paciente y período, no mediciones tomadas en una consulta.

**Merge 2: tabla combinada → patients**
* **Validación de cardinalidad:** Pasó exitosamente (`m:1`), manteniendo las 16,964,801 filas.
* **Auditoría de cruce:** Las mismas **625,620 observaciones (3.69 %) quedan sin paciente**. A diferencia del lab 2, aquí el merge no fue 100 % `both`, por una decisión de carga: `PATIENT` no se leyó de `observations.csv` (sección 1), así que la llave del merge 2 viene de `encounters` y las observaciones sin visita no tienen con qué cruzar.
* **Por qué se acepta:** las filas perdidas son solo DALY, QALY y QOLS, que las secciones 5 y 6 no usan (trabajan con creatinina y signos vitales). **Costo conocido:** si se quisiera analizar esos indicadores por paciente, habría que agregar `PATIENT` a `usecols` en la sección 1.

## 5. Preguntas clínicas

In [7]:
# --- Sección 5: preguntas clínicas ----------------------------------------------
# Parte A (del lab 2): demografía, encuentros por paciente, top 10 observaciones.
# Parte B (nueva): eGFR con calcular_egfr_ckd_epi y grupos etarios con
#                  clasificar_grupo_etario, las funciones probadas del paquete.

# ===== Parte A =====

# A1) Pacientes por RACE y GENDER.
# Se cuenta sobre `pacientes` (una fila por paciente), no sobre tabla_completa:
# ahí cada paciente aparece miles de veces (una por observación) y habría que
# quitar duplicados primero. Contar en la tabla correcta evita ese paso.
# .size() cuenta filas por grupo; .unstack() pasa GENDER de filas a columnas
# para leerlo como tabla cruzada.
tabla_demografia = pacientes.groupby(["RACE", "GENDER"]).size().unstack()
display(tabla_demografia)

# A2) Encuentros por paciente: media y mediana.

visitas_por_paciente = encuentros.groupby("PATIENT").size()
print(
    f"Encuentros por paciente -> Media: {visitas_por_paciente.mean():.2f}, Mediana: {visitas_por_paciente.median():.0f}"
)


# A3) Top 10 observaciones más frecuentes (CODE + DESCRIPTION).

top_observaciones = observaciones[["CODE", "DESCRIPTION"]].value_counts().head(10)
display(top_observaciones)
# ===== Parte B: eGFR =====

# B1) Filas de creatinina. Se filtra por CODE (LOINC), no por DESCRIPTION: es
# estable aunque cambie la redacción del texto (lo decidimos en la sección 1).
CODIGO_CREATININA = "2160-0"  # Creatinine [Mass/volume] in Serum or Plasma
# .copy(): creatinina es un recorte de tabla_completa; sin copy, al agregarle
# columnas pandas avisa (SettingWithCopyWarning / Copy-on-Write) porque no sabe
# si querías modificar también la tabla original.
creatinina = tabla_completa[tabla_completa["CODE"] == CODIGO_CREATININA].copy()
print(f"Mediciones de creatinina: {len(creatinina):,}")

unidades_creatinina = creatinina["UNITS"].value_counts()
print("Distribución de unidades:")
print(unidades_creatinina[unidades_creatinina > 0])

# B2) Preparar las 3 entradas de la función, como columnas de `creatinina`.

creatinina["creatinina_mg_dl"] = pd.to_numeric(creatinina["VALUE"], errors="coerce")
display(creatinina["creatinina_mg_dl"].describe())

# Cálculo de edad al momento de la toma alineando UTC (Opción A)
fecha_observacion = creatinina["DATE"].dt.tz_localize(None)
creatinina["edad_anios"] = (
    fecha_observacion - creatinina["BIRTHDATE"]
).dt.days / 365.25


# B3) Aplicar la función fila por fila.
# calcular_egfr_ckd_epi recibe UN valor de cada cosa (float, float, str), no
# columnas completas. zip() recorre las tres columnas en paralelo y entrega una
# tupla (creatinina, edad, sexo) por fila; la list comprehension llama a la
# función con cada tupla. (~125 mil filas: tarda unos segundos, está bien.)
creatinina["egfr"] = [
    calcular_egfr_ckd_epi(cr, edad, sexo)
    for cr, edad, sexo in zip(
        creatinina["creatinina_mg_dl"], creatinina["edad_anios"], creatinina["GENDER"]
    )
]

# B4) Grupo etario de cada medición, con la función del paquete.
# .map(funcion) aplica una función de UN argumento a cada valor de la columna.

creatinina["grupo_etario"] = creatinina["edad_anios"].map(clasificar_grupo_etario)

# B5) Resumen: eGFR por grupo etario.
# La ecuación CKD-EPI (tanto la versión 2009 como la 2021) fue derivada y validada
# exclusivamente en cohortes de 18 años o más. En pacientes menores de 18 años, la masa muscular y
# la tasa de filtración glomerular divergen de los modelos exponenciales de adultos; aplicarles CKD-EPI
# sobreestima o distorsiona severamente la función renal real (el estándar pediátrico clínico es la ecuación
# bedside de Schwartz basada en talla y creatinina).
#
creatinina_adultos = creatinina[creatinina["edad_anios"] >= 18].copy()

resumen_egfr = (
    creatinina_adultos.groupby("grupo_etario", observed=False)["egfr"]
    .agg(["count", "median", "mean"])
    .round(2)
)

print(
    f"Mediciones pediátricas excluidas (< 18 años): {(creatinina['edad_anios'] < 18).sum():,}"
)
print(f"Mediciones adultas analizadas: {len(creatinina_adultos):,}")
display(resumen_egfr)

GENDER,F,M
RACE,,
asian,798,724
black,1030,938
hawaiian,149,128
native,52,54
other,124,126
white,9428,9337


Encuentros por paciente -> Media: 58.15, Mediana: 36


CODE     DESCRIPTION                                                                                                  
72514-3  Pain severity - 0-10 verbal numeric rating [Score] - Reported                                                    547953
8462-4   Diastolic Blood Pressure                                                                                         325820
8480-6   Systolic Blood Pressure                                                                                          325820
29463-7  Body Weight                                                                                                      309854
8867-4   Heart rate                                                                                                       302920
9279-1   Respiratory rate                                                                                                 302920
8302-2   Body Height                                                                                       

Mediciones de creatinina: 124,686
Distribución de unidades:
UNITS
mg/dL    124686
Name: count, dtype: int64


count    124686.0
mean     2.012463
std       0.74135
min           0.4
25%           1.9
50%           2.0
75%           2.1
max          79.4
Name: creatinina_mg_dl, dtype: Float64

Mediciones pediátricas excluidas (< 18 años): 420
Mediciones adultas analizadas: 124,266


,count,median,mean
grupo_etario,,,
adulto,55428,34.88,37.45
adulto mayor,68838,29.14,31.52


### 5. Análisis Clínico y Justificaciones

**1. Justificación de grupos demográficos:**
Se agrupó a los pacientes utilizando `RACE` y `GENDER`. Esta división permite observar los sesgos inherentes de la muestra sintética (predominantemente caucásica, reflejando el censo base de Synthea). No se utilizó `ETHNICITY` como agrupación principal porque en este estándar suele ser una variable binaria (Hispano/No Hispano), lo que oculta la diversidad poblacional real.

**2. Decisión: Media Global vs Media de Medias:**
Para responder a preguntas clínicas poblacionales, **se debe utilizar la Media de Medias**. 
*Justificación:* El gran cambio entre la media (58.15) y la mediana (36) de encuentros por paciente demuestra que una minoría de pacientes crónicos acude al hospital con extrema frecuencia. Si calculamos la media global de una prueba de laboratorio, esos pacientes crónicos aportarán cientos de mediciones, sesgando el promedio hacia la patología. Al calcular primero el promedio por paciente, y luego promediar la población general, le otorgamos el mismo peso estadístico a un individuo sano que a uno crónico.

*Nota de método:* el lab 2 contaba solo los encuentros que tenían observaciones (`nunique` de `ENCOUNTER` en la tabla unida); aquí se cuentan todas las visitas directamente en `encounters`, donde cada fila es un encuentro. Por eso, y porque los datos se regeneraron, las cifras cambian; la conclusión (media muy por encima de la mediana) se mantiene.

**3. eGFR (CKD-EPI 2021) por grupo etario:**

*Exclusión de pacientes pediátricos (420 mediciones):* La ecuación CKD-EPI (tanto la versión 2009 como la 2021) fue derivada y validada exclusivamente en cohortes de 18 años o más. En pacientes menores de 18 años, la masa muscular y la tasa de filtración glomerular divergen de los modelos exponenciales de adultos; aplicarles CKD-EPI sobreestima o distorsiona severamente la función renal real (el estándar pediátrico clínico es la ecuación bedside de Schwartz basada en talla y creatinina).

*Resultados (124,266 mediciones de adultos):* eGFR mediano de 34.88 mL/min/1.73 m² en adultos (18-64 años) y 29.14 en adultos mayores (≥ 65 años). Ambos valores caen en rango de enfermedad renal crónica moderada a grave (estadio 3b: 30-44; estadio 4: 15-29).

*¿Error de cálculo o característica de los datos?* El eGFR bajo es coherente con las entradas: la creatinina mediana es 2.0 mg/dL (rango de referencia aproximado en adultos: 0.6-1.2 mg/dL), así que la fórmula está devolviendo lo que corresponde a esos valores. La explicación más probable es un **sesgo de selección**: en Synthea la creatinina no se mide a toda la población, sino principalmente a pacientes cuyos módulos clínicos la solicitan (por ejemplo, diabetes o enfermedad renal). Por lo tanto, estos resultados describen **a los pacientes a quienes se les midió creatinina, no a la población general**, y no deben leerse como la prevalencia de enfermedad renal en la muestra.

*Valores extremos de creatinina (revisados en la sección 6):* hay 23 mediciones por encima de 20 mg/dL. Solo las 3 de 79.4 mg/dL superan el máximo documentado en la literatura y se consideran implausibles; excluirlas no cambia la mediana de eGFR (31.25 mL/min/1.73 m² en adultos, diferencia de 0.0003).

## 6. Formato ancho y valores implausibles

### 6. Formato Ancho y Control de Valores Implausibles

**1. El comportamiento de `pivot_table` con medidas repetidas:**
Al reestructurar el dataset a formato ancho (paciente y fecha en filas, analitos en columnas), `pivot_table` necesita una regla para las celdas que reciben más de un valor; aquí se usó la **media** (`aggfunc="mean"`). Como `DATE` incluye la hora al segundo, una celda solo recibe dos valores si hay dos mediciones del mismo analito, del mismo paciente, **en el mismo segundo exacto**. Se encontraron **62 casos**, todos pares: **40 repiten el mismo valor** (registro doble; promediar no cambia nada) y **22 tienen valores distintos** (20 de frecuencia cardíaca, por ejemplo 85.3 y 177.8 lpm). En esos 22, la media fabrica un valor intermedio que nadie midió y esconde la discordancia. Son pocos frente a 954,560 mediciones, así que no alteran el análisis poblacional, pero deben reportarse como inconsistencias de captura y no promediarse en silencio.

**2. Detección de Valores Implausibles:**
Se evaluaron los límites compatibles con parámetros fisiológicos reales.
* **Presión Sistólica:** Se definió como implausible cualquier valor `< 40 mmHg` o `> 260 mmHg`. *(Referencia: Guyton & Hall, Tratado de Fisiología Médica. Valores fuera de este rango en un entorno no crítico de UCI son incompatibles con la vida o indican un error del esfigmomanómetro).*
* **Frecuencia Cardíaca:** Se definió como implausible `< 20 lpm` o `> 250 lpm`. *(Referencia: AHA Guidelines for CPR and ECC).*
* **Creatinina sérica:** Se definió como implausible `< 0.2 mg/dL` o `> 73.8 mg/dL`. El límite superior es el valor más alto publicado en un paciente que sobrevivió; el mismo artículo señala que no se conoce una creatinina incompatible con la vida. *(Referencia: Persaud C, et al. Highest Recorded Serum Creatinine. Case Rep Nephrol. 2021;2021:6048919. doi: 10.1155/2021/6048919).* Por eso los valores de 23.8 a 61.9 mg/dL (20 mediciones) se conservan como **extremos pero posibles** (compatibles con falla renal avanzada), y solo las 3 de 79.4 mg/dL se marcan como implausibles. El límite inferior no marca ninguna fila: el mínimo observado es 0.4 mg/dL.

**3. Acción correctiva (¿Qué hacer con ellos?):**
Con estos rangos se encontraron **3 presiones sistólicas implausibles** y **0 frecuencias cardíacas implausibles**. Incluso en un dataset sintético aparecen valores fuera de rango, así que la auditoría no puede darse por innecesaria. La política aplicada no elimina la fila entera, porque se perderían las mediciones válidas de los otros analitos de ese momento: se reemplaza solo la celda anómala por `NaN` (queda fuera de los estadísticos descriptivos) y se agrega una columna de bandera (`<analito>_es_implausible`) para conservar la trazabilidad de qué dato se corrigió.


In [8]:
# --- Sección 6: formato ancho y valores implausibles -------------------------------
# Pasos: 1) filtrar signos vitales, 2) pasarlos a formato ancho (una columna por
# analito), 3) marcar implausibles con marcar_valores_implausibles (del paquete),
# 4) decidir qué hacer con ellos, 5) revisar la creatinina de 79.4 mg/dL.

# 1) Signos vitales, filtrados por CODE (LOINC), igual que la creatinina.
CODIGOS_VITALES = {
    "8480-6": "Systolic Blood Pressure",
    "8462-4": "Diastolic Blood Pressure",
    "8867-4": "Heart rate",
}
# .isin(lista) = True si el CODE está en la lista. list(dict) da las llaves.
vitales = tabla_completa[tabla_completa["CODE"].isin(list(CODIGOS_VITALES))].copy()


filas_vitales_inicio = len(vitales)
vitales["valor"] = pd.to_numeric(vitales["VALUE"], errors="coerce")
vitales = vitales.dropna(subset=["valor"])
filas_descartadas = filas_vitales_inicio - len(vitales)
print(
    f"Filas no numéricas descartadas en signos vitales: {filas_descartadas:,} (restan {len(vitales):,})"
)

# 2) Formato ancho: verificación e inspección de duplicados
cols_duplicado = ["PATIENT", "DATE", "CODE"]
duplicados_mask = vitales.duplicated(subset=cols_duplicado, keep=False)
conteo_duplicados = vitales.duplicated(subset=cols_duplicado).sum()

print(f"Combinaciones repetidas (PATIENT, DATE, CODE): {conteo_duplicados}")

if conteo_duplicados > 0:
    filas_repetidas = vitales[duplicados_mask].sort_values(by=cols_duplicado)

    valores_distintos_por_grupo = filas_repetidas.groupby(
        cols_duplicado, observed=True
    )["valor"].nunique()
    pares_discordantes = (valores_distintos_por_grupo > 1).sum()

    if pares_discordantes == 0:
        print(
            "-> Hallazgo: Todas las repeticiones son registros redundantes con valores idénticos. "
            "aggfunc='mean' no altera los datos."
        )
    else:
        print(
            f"-> Hallazgo: {pares_discordantes} combinaciones tienen valores distintos (discordantes). "
            "aggfunc='mean' promedia lecturas no idénticas registradas en el mismo segundo."
        )

    display(filas_repetidas[["PATIENT", "DATE", "DESCRIPTION", "valor"]].head(6))
else:
    print(
        "-> Hallazgo: No hay múltiples tomas en el mismo segundo exacto. "
        "aggfunc='mean' no está promediando valores repetidos."
    )

# --- Construcción de vitales_ancho ---
vitales_ancho = vitales.pivot_table(
    index=["PATIENT", "DATE"],
    columns="DESCRIPTION",
    values="valor",
    aggfunc="mean",
    observed=True,
)
display(vitales_ancho.head())

# 3) Marcar implausibles con la función del paquete (rangos del markdown).
# (analito: (mínimo, máximo)). Unidades: mmHg para presión, lpm para FC.
RANGOS_PLAUSIBLES = {
    "Systolic Blood Pressure": (40, 260),
    "Heart rate": (20, 250),
}

mascaras_implausibles = {}
for analito, (minimo, maximo) in RANGOS_PLAUSIBLES.items():
    if analito in vitales_ancho.columns:
        mascara = marcar_valores_implausibles(vitales_ancho[analito], minimo, maximo)
        mascaras_implausibles[analito] = mascara
        print(
            f"Valores implausibles en '{analito}' (límites {minimo}-{maximo}): {mascara.sum():,}"
        )

# 4) ¿Qué hacer con los implausibles? El markdown recomienda poner la celda en
#    NaN (no borrar la fila) y dejar una bandera.

vitales_ancho_limpio = vitales_ancho.copy()

for analito, mascara in mascaras_implausibles.items():
    if mascara.sum() > 0:
        # Creamos bandera booleana para preservar la trazabilidad del dato auditado
        vitales_ancho_limpio[f"{analito}_es_implausible"] = mascara
        # Reemplazamos el valor absurdo por NaN sin eliminar las otras mediciones válidas de la fila
        vitales_ancho_limpio[analito] = vitales_ancho_limpio[analito].mask(mascara)
        print(
            f"-> Se reemplazaron {mascara.sum()} celdas implausibles por NaN en '{analito}'."
        )
    else:
        print(
            f"-> 0 valores implausibles detectados en '{analito}'; no se modificaron valores."
        )

# 5) Creatinina: plausibilidad fisiológica con respaldo bibliográfico
# Referencia:  Persaud C, Sandesara U, Hoang V, Tate J, Latack W, Dado D. Highest Recorded Serum Creatinine.
# Case Reports in Nephrology. 2021;2021:6048919. doi: 10.1155/2021/6048919
# Se define [0.2, 73.8] mg/dL como intervalo biológico compatible con muestras clínicas no diluidas.
LIMITE_MIN_CR = 0.2  # mg/dL
LIMITE_MAX_CR = 73.8  # mg/dL

mascara_cr_implausible = marcar_valores_implausibles(
    creatinina["creatinina_mg_dl"], LIMITE_MIN_CR, LIMITE_MAX_CR
)
print(
    f"Mediciones de creatinina implausibles (> {LIMITE_MAX_CR} mg/dL): {mascara_cr_implausible.sum():,}"
)

# Inspección de los valores atípicos detectados
print("\nDistribución de las mediciones fuera de rango:")
display(creatinina.loc[mascara_cr_implausible, "creatinina_mg_dl"].value_counts())

# Impacto en la mediana de eGFR en la población adulta (>= 18 años)
egfr_con_outliers = creatinina[creatinina["edad_anios"] >= 18]["egfr"].median()
egfr_sin_outliers = creatinina[
    (creatinina["edad_anios"] >= 18) & (~mascara_cr_implausible)
]["egfr"].median()

print(f"\nMediana eGFR con valores crudos:     {egfr_con_outliers:.2f} mL/min/1.73m²")
print(f"Mediana eGFR sin outliers extremos:  {egfr_sin_outliers:.2f} mL/min/1.73m²")
print(f"Diferencia: {abs(egfr_con_outliers - egfr_sin_outliers):.4f} mL/min/1.73m²")

Filas no numéricas descartadas en signos vitales: 0 (restan 954,560)
Combinaciones repetidas (PATIENT, DATE, CODE): 62
-> Hallazgo: 22 combinaciones tienen valores distintos (discordantes). aggfunc='mean' promedia lecturas no idénticas registradas en el mismo segundo.


,PATIENT,DATE,DESCRIPTION,valor
10094111,074d4588-af63-2af9-5322-184b7b8c3d01,2020-10-18 23:36:25+00:00,Diastolic Blood Pressure,68.0
10094413,074d4588-af63-2af9-5322-184b7b8c3d01,2020-10-18 23:36:25+00:00,Diastolic Blood Pressure,68.0
10094113,074d4588-af63-2af9-5322-184b7b8c3d01,2020-10-18 23:36:25+00:00,Systolic Blood Pressure,109.0
10094414,074d4588-af63-2af9-5322-184b7b8c3d01,2020-10-18 23:36:25+00:00,Systolic Blood Pressure,109.0
10094108,074d4588-af63-2af9-5322-184b7b8c3d01,2020-10-18 23:36:25+00:00,Heart rate,177.8
10094411,074d4588-af63-2af9-5322-184b7b8c3d01,2020-10-18 23:36:25+00:00,Heart rate,85.3


DESCRIPTION                                                     Diastolic Blood Pressure  \
PATIENT                              DATE                                                  
000085c1-5b07-25b7-26cc-e0639d7f42d4 2016-09-19 18:31:57+00:00                      79.0   
                                     2019-09-23 18:31:57+00:00                      76.0   
                                     2020-05-25 18:31:57+00:00                      76.0   
                                     2022-07-11 18:31:57+00:00                      68.0   
                                     2024-07-15 18:31:57+00:00                      76.0   

DESCRIPTION                                                     Heart rate  \
PATIENT                              DATE                                    
000085c1-5b07-25b7-26cc-e0639d7f42d4 2016-09-19 18:31:57+00:00        91.0   
                                     2019-09-23 18:31:57+00:00        90.0   
                                     2020-05-25 18:31:57+00:00        66.0   
                                     2022-07-11 18:31:57+00:00        72.0   
                                     2024-07-15 18:31:57+00:00       100.0   

DESCRIPTION                                                     Systolic Blood Pressure  
PATIENT                              DATE                                                
000085c1-5b07-25b7-26cc-e0639d7f42d4 2016-09-19 18:31:57+00:00                    115.0  
                                     2019-09-23 18:31:57+00:00                    111.0  
                                     2020-05-25 18:31:57+00:00                    113.0  
                                     2022-07-11 18:31:57+00:00                    116.0  
                                     2024-07-15 18:31:57+00:00                    102.0

Valores implausibles en 'Systolic Blood Pressure' (límites 40-260): 3
Valores implausibles en 'Heart rate' (límites 20-250): 0
-> Se reemplazaron 3 celdas implausibles por NaN en 'Systolic Blood Pressure'.
-> 0 valores implausibles detectados en 'Heart rate'; no se modificaron valores.
Mediciones de creatinina implausibles (> 73.8 mg/dL): 3

Distribución de las mediciones fuera de rango:


creatinina_mg_dl
79.4    3
Name: count, dtype: Int64


Mediana eGFR con valores crudos:     31.25 mL/min/1.73m²
Mediana eGFR sin outliers extremos:  31.25 mL/min/1.73m²
Diferencia: 0.0003 mL/min/1.73m²


## 7. Recomendación técnica final

### Recomendación técnica final

**¿Qué herramienta usaría para este volumen de datos y por qué?**
Para este volumen (unos 17 millones de observaciones, un CSV de varios GB, una sola máquina), **pandas es suficiente, siempre que los tipos se declaren desde la carga**. En este notebook, el cuello de botella no fue la librería, sino el **pico de memoria** al leer: leer `observations.csv` sin tipos habría construido primero la versión completa como texto. Al pedir solo las columnas necesarias (`usecols`) y sus tipos finales (`dtype`, `parse_dates`) en `read_csv`, la tabla quedó en 1,285 MB, y convertir `encounters` a `category` y fechas redujo su memoria en un 83 % (sección 2). Con esas decisiones, todo el análisis corre en memoria sin necesitar otra herramienta.

Lo que hace confiable este análisis tampoco depende de la librería: la lógica (tipos, auditoría, uniones, clasificación, eGFR) vive en funciones del paquete `clinlab`, probadas con `pytest`, y el notebook solo las llama.

**¿A partir de qué punto cambiaría de opinión?**

1. **Si los datos, ya optimizados, no caben cómodamente en RAM, o si el análisis se repite muchas veces:** pasaría a **Polars** o **DuckDB**. Ambos trabajan por columnas y con ejecución perezosa (*lazy*): planifican la consulta completa y leen del disco solo las columnas y filas que la respuesta necesita, en vez de cargar la tabla entera. Un primer paso intermedio, sin cambiar de librería, sería guardar los CSV como **Parquet**, que conserva los tipos y permite leer columnas sueltas mucho más rápido.
2. **Si los datos escalan a varias máquinas (terabytes):** pasaría a **PySpark**, cuya complejidad solo se justifica cuando una sola máquina ya no alcanza y hace falta distribuir el trabajo en un clúster.
3. **Si los datos son chicos:** pandas sin más, porque su ecosistema y la facilidad para explorar siguen siendo su mayor ventaja.

*Nota:* el notebook del laboratorio 2 incluía un benchmark comparativo entre pandas, Polars y PySpark. No se repite aquí porque este notebook se centra en el análisis con funciones probadas; esta recomendación se apoya solo en lo medido en este notebook.